# Inkengine CUDA CycleAudit gate (M24)

Goal: on a Kaggle 2xT4 runner, time the synthetic 3-pass MoE GGUF with RORET
off and on, then drive the real loopkit audit CLI through
tools/audit/run_gpu_cycle_audit.py. This closes the M9 blocker recorded in
DECISIONS.tsv.

Honest constraints:
- Synthetic fixture only; no real weights. Run at most one kernel.
- No speedup without a rendered gate whose claim <= ci_low.
- repeats >= 20 (50 here), loops = 3, warmup = 5, seed 0, 2000 bootstrap resamples.
- The pre-registered claim is 1.0, frozen before this run.
- T4 (sm_75) has no FP4 or FP8 tensor cores; those are storage formats only.
- RORET_PASS and MoE bookkeeping counts are counts, not speedups.
- NO SPEED CLAIM unless claim_valid and claim <= ci_low.
- overall_verdict PASS means the gate ran with integrity, not that a speed claim exists.
- Fail-closed: any stage failure or a missing M24_SUMMARY_JSON line gives NO_GO.


In [ ]:
import os, sys, subprocess, json, hashlib, time
from pathlib import Path

WORK = "/kaggle/working"
REPO = WORK + "/inkengine-llama"
LOOPKIT = WORK + "/loopkit"
GGUF = WORK + "/synthetic.gguf"
OUT = WORK + "/gpu-cycle-audit"
AUDIT_LOG = WORK + "/gpu_cycle_audit.log"
HOOK_LINE = "RORET union-resident MoE bookkeeping enabled"
SPEED_GATE = "CycleAudit claim <= ci_low with repeats >= 20 and loops = 3"

def run_logged(cmd, log_path, cwd=None, env=None, timeout=None):
    # timeout is None for clone, build, fixture and audit. Never cap the CUDA build.
    try:
        with open(log_path, "w") as f:
            p = subprocess.run(cmd, cwd=cwd, env=env, stdout=f,
                               stderr=subprocess.STDOUT, timeout=timeout)
        return p.returncode
    except subprocess.TimeoutExpired:
        with open(log_path, "a") as f:
            f.write("\nTIMEOUT: command exceeded " + str(timeout) + " seconds\n")
        return 124

def tail(path, n=40):
    try:
        lines = open(path, errors="replace").read().splitlines()
    except OSError:
        return ""
    return "\n".join(lines[-n:])

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

summary = {
    "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo_head": "",
    "loopkit_head": "",
    "nvcc_version": "",
    "deps_ok": False,
    "clone_ok": False,
    "configure_exit": None,
    "build_exit": None,
    "build_ok": False,
    "fixture_sha256": "",
    "audit_command": [],
    "audit_exit": None,
    "audit_log_tail": "",
    "m24": None,
    "speed_claim": "",
    "speed_gate": SPEED_GATE,
    "blocker": "",
    "overall_verdict": "NO_GO",
}

try:
    import numpy, scipy
    summary["deps_ok"] = True
except ImportError:
    rc = run_logged([sys.executable, "-m", "pip", "install", "numpy", "scipy"],
                    WORK + "/pip_deps.log")
    summary["deps_ok"] = (rc == 0)
    print("pip install numpy scipy exit", rc)
print("python", sys.version.split()[0], "deps_ok", summary["deps_ok"])

try:
    r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    summary["nvcc_version"] = (r.stdout or r.stderr).strip()
except OSError:
    summary["nvcc_version"] = "BLOCKER: no nvcc"
print(summary["nvcc_version"])

PIPELINE_OK = summary["deps_ok"] and "BLOCKER" not in summary["nvcc_version"]
if not PIPELINE_OK:
    summary["blocker"] = "python deps or nvcc unavailable"
    print("BLOCKER:", summary["blocker"])


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    if not os.path.isdir(REPO):
        cmd = ["git", "clone", "-b", "main",
               "https://github.com/10xdev4u-alt/inkengine-llama.git", REPO]
        print("RUN", " ".join(cmd))
        if run_logged(cmd, WORK + "/clone_repo.log") != 0:
            summary["blocker"] = "git clone fork failed"
    else:
        print("clone: existing dir, skip", REPO)
    if not os.path.isdir(LOOPKIT):
        cmd = ["git", "clone",
               "https://github.com/10xdev4u-alt/loopkit.git", LOOPKIT]
        print("RUN", " ".join(cmd))
        if run_logged(cmd, WORK + "/clone_loopkit.log") != 0:
            summary["blocker"] = "git clone loopkit failed"
    else:
        print("clone: existing dir, skip", LOOPKIT)
    if summary["blocker"] == "":
        for label, d in (("repo", REPO), ("loopkit", LOOPKIT)):
            r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=d,
                               capture_output=True, text=True)
            head = r.stdout.strip()
            summary[label + "_head"] = head
            print(label, "HEAD", head)
            if not head:
                summary["blocker"] = label + " rev-parse HEAD failed"
        if not os.path.isfile(LOOPKIT + "/src/loopkit/cli.py"):
            summary["blocker"] = "loopkit cli.py missing after clone"
    summary["clone_ok"] = (summary["blocker"] == "")
    PIPELINE_OK = summary["clone_ok"]
    if not PIPELINE_OK:
        print("BLOCKER:", summary["blocker"])


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    cfg = ["cmake", "-S", ".", "-B", "build", "-DGGML_CUDA=ON",
           "-DCMAKE_CUDA_ARCHITECTURES=75", "-DLLAMA_BUILD_TESTS=OFF",
           "-DLLAMA_CURL=OFF", "-DGGML_CUDA_NO_VMM=ON"]
    print("RUN", " ".join(cfg))
    rc1 = run_logged(cfg, WORK + "/cmake_configure.log", cwd=REPO)
    summary["configure_exit"] = rc1
    print("configure exit", rc1)
    print("configure log tail:")
    print(tail(WORK + "/cmake_configure.log", n=20))
    bld = ["cmake", "--build", "build", "--target", "llama-cli", "-j4"]
    print("RUN", " ".join(bld))
    rc2 = run_logged(bld, WORK + "/cmake_build.log", cwd=REPO)
    summary["build_exit"] = rc2
    print("build exit", rc2)
    print("build log tail:")
    print(tail(WORK + "/cmake_build.log", n=20))
    summary["build_ok"] = (rc1 == 0 and rc2 == 0 and
                           os.path.isfile(REPO + "/build/bin/llama-cli"))
    PIPELINE_OK = summary["build_ok"]
    if not PIPELINE_OK:
        summary["blocker"] = "CUDA build failed or llama-cli missing"
        print("BLOCKER:", summary["blocker"])


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    gen_env = dict(os.environ)
    gen_env["PYTHONPATH"] = os.pathsep.join(
        [REPO + "/gguf-py", gen_env.get("PYTHONPATH", "")]).rstrip(os.pathsep)
    cmd = [sys.executable, REPO + "/tools/audit/make_synthetic_gguf.py",
           "--seed", "0", "--out", GGUF]
    print("RUN", " ".join(cmd))
    rc = run_logged(cmd, WORK + "/make_gguf.log", cwd=REPO, env=gen_env)
    print("generator exit", rc)
    if rc == 0 and os.path.isfile(GGUF) and os.path.getsize(GGUF) > 0:
        summary["fixture_sha256"] = sha256_file(GGUF)
        print("fixture_sha256", summary["fixture_sha256"])
    else:
        summary["blocker"] = "GGUF generation failed or file missing/empty"
        PIPELINE_OK = False
        print("BLOCKER:", summary["blocker"])


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    audit_env = dict(os.environ)
    audit_env["PYTHONPATH"] = os.pathsep.join(
        [LOOPKIT + "/src", audit_env.get("PYTHONPATH", "")]).rstrip(os.pathsep)
    cmd = [sys.executable, REPO + "/tools/audit/run_gpu_cycle_audit.py",
           "--llama-cli", REPO + "/build/bin/llama-cli",
           "--fixture", GGUF, "--out", OUT,
           "--loopkit-src", LOOPKIT + "/src",
           "--repeats", "50", "--warmup", "5", "--loops", "3",
           "--claim", "1.0", "--cpu-share", "0.0", "--ngl", "99",
           "--moe-cache-mib", "16",
           "--environment-label", "Kaggle 2xT4 CUDA"]
    summary["audit_command"] = cmd
    print("RUN", " ".join(cmd))
    rc = run_logged(cmd, AUDIT_LOG, env=audit_env)
    summary["audit_exit"] = rc
    summary["audit_log_tail"] = tail(AUDIT_LOG, n=60)
    print("audit exit", rc)
    print("audit log tail:")
    print(summary["audit_log_tail"])
    if rc != 0:
        summary["blocker"] = "run_gpu_cycle_audit.py failed"
        PIPELINE_OK = False
        print("BLOCKER:", summary["blocker"])


In [ ]:
marker = "M24_SUMMARY_JSON="
lines = []
try:
    lines = open(AUDIT_LOG, errors="replace").read().splitlines()
except OSError:
    pass
m24 = None
for line in reversed(lines):
    pos = line.find(marker)
    if pos >= 0:
        try:
            m24 = json.loads(line[pos + len(marker):].strip())
        except ValueError:
            m24 = None
        break
summary["m24"] = m24
summary["speed_claim"] = "NO SPEED CLAIM unless claim_valid and claim<=ci_low"
summary["speed_gate"] = SPEED_GATE
good = (summary["build_ok"] and summary["audit_exit"] == 0 and
        isinstance(m24, dict) and
        m24.get("verdict") in ("CLAIM_SUPPORTED", "NO_CLAIM") and
        m24.get("negative_controls_pass") is True and
        summary["blocker"] == "")
summary["overall_verdict"] = "PASS" if good else "NO_GO"
if not good and summary["blocker"] == "":
    summary["blocker"] = "missing or invalid M24_SUMMARY_JSON line"
out_path = WORK + "/m24_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print("overall_verdict", summary["overall_verdict"])
if isinstance(m24, dict):
    print("gate verdict", m24.get("verdict"), "claim", m24.get("claim"),
          "ci_low", m24.get("ci_low"), "claim_valid", m24.get("claim_valid"))
print("PASS means the gate ran with integrity, not that a speed claim exists.")
print("No speed number unless claim_valid and claim <= ci_low with repeats >= 20 and loops = 3.")
print(json.dumps(summary, indent=2))
print("summary written to", out_path)
